# 01 — Data Collection

**Question:** what does the NBA pay for, and who is over- or underpaid?

**Source:** Basketball-Reference.com (Sports Reference LLC), scraped by us — 389 pages, seasons 2015-16 → 2025-26.
Free APIs (balldontlie, nba_api) have no salary data, so we scrape.

| Step | Script | Output |
|---|---|---|
| 1 | `src/download.py` | raw pages (rate-limited, ~45 min) |
| 2 | `src/parse_bref.py` | `data/raw/*.csv` |
| 3 | `src/build_dataset.py` | `data/processed/player_seasons.csv` |

In [1]:
import sys
sys.path.append("../src")
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from features import build_features
import plotting as P

P.setup()
pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if abs(x) >= 1000 else f"{x:,.3f}")

In [2]:
raw = {n: pd.read_csv(f"../data/raw/{n}.csv") for n in
       ["per_game", "advanced", "salaries", "rosters", "teams", "draft", "salary_cap"]}
pd.DataFrame({n: {"rows": len(d)} for n, d in raw.items()}).T

,rows
per_game,7595
advanced,7595
salaries,7299
rosters,6819
teams,330
draft,2101
salary_cap,43


## Cleaning rules
- Tables joined on Basketball-Reference **player id** (no name matching).
- Traded players: keep the combined season line; assign to last team.
- Salary = amount from a team the player **played for** (ignores dead money from waivers).
- &#36;0 listings (two-way contracts with no amount) → missing.
- Target uses **salary ÷ cap**, since the cap grew from &#36;70M to &#36;155M.

In [3]:
ps = pd.read_csv("../data/processed/player_seasons.csv")
assert not ps.duplicated(["season", "player_id"]).any()
print(f"{len(ps):,} player-seasons | {ps.player_id.nunique():,} players | {ps.has_salary.mean():.0%} have a salary")
(ps.groupby("season").agg(players=("player_id", "size"), with_salary=("has_salary", "sum"),
                          median_salary=("salary", "median"), salary_cap=("salary_cap", "first"))
   .style.format({"median_salary": "${:,.0f}", "salary_cap": "${:,.0f}"}))

5,968 player-seasons | 1,554 players | 91% have a salary


,players,with_salary,median_salary,salary_cap
season,,,,
2016,476,476,"$2,568,830","$70,000,000"
2017,486,484,"$3,162,022","$94,143,000"
2018,540,483,"$3,097,800","$99,093,000"
2019,530,482,"$3,260,917","$101,869,000"
2020,529,480,"$3,349,300","$109,140,000"
2021,540,481,"$3,640,200","$109,140,000"
2022,605,562,"$2,641,691","$112,414,000"
2023,539,485,"$4,000,000","$123,655,000"
2024,572,499,"$4,310,250","$136,021,000"
